# Drive Daily Stats — Functional ETL

## Purpose

Transform unified Bronze Backblaze drive observations into clean, typed daily drive records in the Curated/Silver layer.
Invalid records are routed to a dedicated quarantine table for investigation.

**Source:** `backblaze_data.ing_bronze.drive_observations`  

## Targets

**Curated/Silver:** `backblaze_data.sot_silver.drive_daily_stats`  
**Quarantine:** `backblaze_data.sot_silver.drive_stats_quarantine`  

## Processing Responsibilities

1. Read the unified Bronze Delta table.
2. Standardize date values and convert source columns to their target data types.
3. Validate mandatory fields and business rules.
4. Identify duplicate records at the expected grain.
5. Route rejected records to the quarantine table with rejection details.
6. Load accepted records into the Curated/Silver table.
7. Validate the final record counts, schema, and data quality.

## Data Quality Rules

- `_period` must be present.
- `date` must be present and parseable.
- `serial_number` must be present.
- `failure` must be either 0 or 1.
- Non-null capacity values must be convertible to BIGINT.
- Non-null SMART normalized values must be convertible to BIGINT.
- Duplicate groups at `(_period, date, serial_number)` are quarantined.
- Valid date formats include `yyyy-MM-dd` and `M/d/yy`.

**Grain:**  `(_period, date, serial_number)`  
**Load Strategy:**   Full refresh of the two Silver output tables from the current Bronze source snapshot.

## Execution Order

Run the following notebooks before this notebook:
1. `nb_drive_daily_stats_ddl`
2. `nb_drive_stats_quarantine_ddl`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

SOURCE_TABLE = "backblaze_data.ing_bronze.drive_observations"
TARGET_TABLE = "backblaze_data.sot_silver.drive_daily_stats"
QUARANTINE_TABLE = "backblaze_data.sot_silver.drive_stats_quarantine"

print("Source and target configuration loaded.")
print(f"Source: {SOURCE_TABLE}")
print(f"Curated target: {TARGET_TABLE}")
print(f"Quarantine target: {QUARANTINE_TABLE}")

In [0]:
required_tables = [
    SOURCE_TABLE,
    TARGET_TABLE,
    QUARANTINE_TABLE
]

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Required table does not exist: {table_name}")

print("All required tables exist.")

print(f"Bronze source rows: {spark.table(SOURCE_TABLE).count()}")
print(f"Curated target current rows: {spark.table(TARGET_TABLE).count()}")
print(f"Quarantine current rows: {spark.table(QUARANTINE_TABLE).count()}")

In [0]:
source_df = spark.table(SOURCE_TABLE)

required_columns = [
    "date",
    "serial_number",
    "capacity_bytes",
    "failure",
    "_period",
    "_source_file_path",
    "_ingested_at"
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in source_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required Bronze columns: {missing_columns}")

source_columns = source_df.columns
target_columns = spark.table(TARGET_TABLE).columns

if source_columns != target_columns:
    raise ValueError(
        "Bronze and Curated column names/order do not match. "
        "Review the DDL and source schema."
    )

print(f"Bronze columns: {len(source_columns)}")
print(f"Curated columns: {len(target_columns)}")
print("Source and target column structures match.")

In [0]:
# Preserve the original Bronze record for quarantine investigation.
transformed_df = source_df.withColumn(
    "_raw_record_payload",
    F.to_json(
        F.struct(*[
            F.col(f"`{column_name}`")
            for column_name in source_columns
        ])
    )
)

# Preserve original values needed for data-quality checks.
transformed_df = (
    transformed_df
    .withColumn("_raw_date", F.col("date"))
    .withColumn("_raw_failure", F.col("failure"))
    .withColumn("_raw_capacity_bytes", F.col("capacity_bytes"))
)

# Parse both supported date formats.
# try_to_timestamp returns NULL instead of failing for an invalid date.
transformed_df = transformed_df.withColumn(
    "date",
    F.expr(
        """
        COALESCE(
            CAST(try_to_timestamp(`date`, 'yyyy-MM-dd') AS DATE),
            CAST(try_to_timestamp(`date`, 'M/d/yy') AS DATE)
        )
        """
    )
)

# Convert capacity and failure using safe casts.
transformed_df = (
    transformed_df.withColumn(
    "capacity_bytes",
        F.expr(
            "try_cast(try_cast(`capacity_bytes` AS DECIMAL(38,0)) AS BIGINT)"
        )
    )
    .withColumn(
        "failure",
        F.expr("try_cast(`failure` AS INT)")
    )
)

# Identify SMART normalized columns dynamically.
smart_normalized_columns = [
    column_name
    for column_name in source_columns
    if column_name.startswith("smart_")
    and column_name.endswith("_normalized")
]

# Detect malformed non-null SMART normalized values before casting.
invalid_smart_condition = F.lit(False)

for column_name in smart_normalized_columns:
    converted_value = F.expr(
        f"try_cast(`{column_name}` AS BIGINT)"
    )

    invalid_smart_condition = (
        invalid_smart_condition
        | (
            F.col(column_name).isNotNull()
            & converted_value.isNull()
        )
    )

transformed_df = transformed_df.withColumn(
    "_invalid_smart_normalized",
    invalid_smart_condition
)

# Convert all SMART normalized attributes to BIGINT.
# SMART raw attributes remain STRING.
for column_name in smart_normalized_columns:
    transformed_df = transformed_df.withColumn(
        column_name,
        F.expr(f"try_cast(`{column_name}` AS BIGINT)")
    )

print("Date and numeric conversions completed.")
print(f"SMART normalized columns converted: {len(smart_normalized_columns)}")

In [0]:
# Define the expected grain.
grain_columns = [
    "_period",
    "date",
    "serial_number"
]

# Count records sharing the same valid grain.
duplicate_window = Window.partitionBy(*grain_columns)

transformed_df = transformed_df.withColumn(
    "_duplicate_count",
    F.count("*").over(duplicate_window)
)

# Required-field validation.
invalid_period = (
    F.col("_period").isNull()
    | (F.trim(F.col("_period")) == "")
)

invalid_date = (
    F.col("_raw_date").isNull()
    | (F.trim(F.col("_raw_date")) == "")
    | F.col("date").isNull()
)

invalid_serial_number = (
    F.col("serial_number").isNull()
    | (F.trim(F.col("serial_number")) == "")
)

# Failure must be either 0 or 1.
invalid_failure = (
    F.col("_raw_failure").isNull()
    | F.col("failure").isNull()
    | (~F.col("failure").isin(0, 1))
)

# Non-null capacity values must convert successfully.
invalid_capacity = (
    F.col("_raw_capacity_bytes").isNotNull()
    & F.col("capacity_bytes").isNull()
)

# A duplicate is evaluated only when all grain fields are valid.
invalid_duplicate = (
    F.col("_period").isNotNull()
    & F.col("date").isNotNull()
    & F.col("serial_number").isNotNull()
    & (F.col("_duplicate_count") > 1)
)

# Combine all rejection reasons.
rejection_reason = F.concat_ws(
    "; ",
    F.when(invalid_period, F.lit("Missing period")),
    F.when(invalid_date, F.lit("Missing or unparseable date")),
    F.when(invalid_serial_number, F.lit("Missing serial number")),
    F.when(invalid_failure, F.lit("Invalid failure value")),
    F.when(invalid_capacity, F.lit("Invalid capacity_bytes value")),
    F.when(
        F.col("_invalid_smart_normalized"),
        F.lit("Invalid SMART normalized numeric value")
    ),
    F.when(invalid_duplicate, F.lit("Duplicate grain"))
)

# Record the corresponding rule identifiers.
rule_name = F.concat_ws(
    ",",
    F.when(invalid_period, F.lit("DQ_001_PERIOD_REQUIRED")),
    F.when(invalid_date, F.lit("DQ_002_DATE_VALID")),
    F.when(invalid_serial_number, F.lit("DQ_003_SERIAL_REQUIRED")),
    F.when(invalid_failure, F.lit("DQ_004_FAILURE_FLAG")),
    F.when(invalid_capacity, F.lit("DQ_005_CAPACITY_NUMERIC")),
    F.when(
        F.col("_invalid_smart_normalized"),
        F.lit("DQ_006_SMART_NORMALIZED_NUMERIC")
    ),
    F.when(invalid_duplicate, F.lit("DQ_007_UNIQUE_GRAIN"))
)

classified_df = (
    transformed_df
    .withColumn("_rejection_reason", rejection_reason)
    .withColumn("_rule_name", rule_name)
    .withColumn(
        "_is_rejected",
        F.col("_rejection_reason") != ""
    )
)

print("Data-quality rules applied.")

In [0]:
quarantine_df = (
    classified_df
    .filter(F.col("_is_rejected"))
    .select(
        F.coalesce(
            F.col("_raw_record_payload"),
            F.lit("{}")
        ).alias("record_payload"),

        # Use UNKNOWN if the source period itself is missing.
        # The original NULL value remains visible in record_payload.
        F.coalesce(
            F.col("_period"),
            F.lit("UNKNOWN")
        ).alias("_period"),

        F.col("_source_file_path"),
        F.col("_ingested_at"),

        F.coalesce(
            F.col("_rejection_reason"),
            F.lit("Unspecified rejection")
        ).alias("rejection_reason"),

        F.coalesce(
            F.col("_rule_name"),
            F.lit("DQ_UNKNOWN")
        ).alias("rule_name"),

        F.current_timestamp().alias("quarantined_at")
    )
)

# Full refresh: prevents duplicate quarantine entries on reruns.
(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table refreshed: {QUARANTINE_TABLE}")

In [0]:
accepted_df = (
    classified_df
    .filter(~F.col("_is_rejected"))
    .select(*target_columns)
)

# Full refresh of the Curated/Silver table.
(
    accepted_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

print(f"Curated table refreshed: {TARGET_TABLE}")

In [0]:
source_count = spark.table(SOURCE_TABLE).count()

curated_df = spark.table(TARGET_TABLE)
quarantine_result_df = spark.table(QUARANTINE_TABLE)

curated_count = curated_df.count()
quarantine_count = quarantine_result_df.count()

print("========== ETL VALIDATION ==========")

print(f"Bronze source records : {source_count:,}")
print(f"Curated accepted      : {curated_count:,}")
print(f"Quarantined rejected  : {quarantine_count:,}")

print("-----------------------------------")

if source_count != curated_count + quarantine_count:
    raise ValueError(
        "Record reconciliation failed. "
        "Source count does not equal accepted plus rejected records."
    )

print("Record reconciliation: PASSED")

# Check the expected grain in the Curated table.
duplicate_count = (
    curated_df
    .groupBy(*grain_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_count != 0:
    raise ValueError(
        f"Curated grain validation failed. Duplicate groups: {duplicate_count}"
    )

print("Curated grain validation: PASSED")

# Check mandatory fields in the Curated table.
invalid_curated_count = curated_df.filter(
    F.col("_period").isNull()
    | F.col("date").isNull()
    | F.col("serial_number").isNull()
    | (~F.col("failure").isin(0, 1))
).count()

if invalid_curated_count != 0:
    raise ValueError(
        f"Curated data-quality validation failed: {invalid_curated_count}"
    )

print("Curated mandatory-field validation: PASSED")

print("===================================")

In [0]:
print("CURATED RECORDS BY PERIOD")

display(
    spark.table(TARGET_TABLE)
    .groupBy("_period")
    .agg(
        F.count("*").alias("accepted_records"),
        F.countDistinct("serial_number").alias("distinct_drives"),
        F.min("date").alias("minimum_date"),
        F.max("date").alias("maximum_date")
    )
    .orderBy("_period")
)

print("QUARANTINED RECORDS BY PERIOD")

display(
    spark.table(QUARANTINE_TABLE)
    .groupBy("_period")
    .count()
    .orderBy("_period")
)